<a href="https://colab.research.google.com/github/datasentient1/RLHF-Logic-Verification-Framework/blob/master/notebooks/colab_mvp_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Verifier-Guided Reasoning MVP (Weeks 1-8)

This notebook is organized by roadmap week and designed as an employer-facing artifact for **data annotation + curation** roles.

What this notebook demonstrates:
- deterministic verifier-guided quality control for reasoning traces,
- reviewer controls with accept/reject/fix decisions,
- measurable best-of-N and rejection-sampling behavior,
- a narrow logic extension evaluated **separately** from arithmetic,
- reproducibility gates before publication artifacts.


## Section 0 (Project Explanation)

A full technical-but-accessible explanation is available in:
- `docs/section_0_project_explanation.md`

Read it before demoing if your audience is not deeply ML-specialized.


In [ ]:
import os
import sys
import json
from pathlib import Path
from collections import Counter, defaultdict
from pprint import pprint

IN_COLAB = 'google.colab' in sys.modules
print({'in_colab': IN_COLAB, 'python': sys.version.split()[0]})

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

print('pandas:', 'ok' if pd is not None else 'missing')
print('matplotlib:', 'ok' if plt is not None else 'missing')


## Weeks 1-2: Foundation Check

Goal: prove the base arithmetic pipeline is reproducible and trackable.


In [ ]:
# Week 1-2 / Step 1: clone + install

repo_name = 'RLHF-Logic-Verification-Framework'
repo_https = 'https://github.com/datasentient1/RLHF-Logic-Verification-Framework.git'

if IN_COLAB:
    workspace = Path('/content')
    repo_root = workspace / repo_name
    if not repo_root.exists():
        print(f'Cloning {repo_https} ...')
        get_ipython().system(f'git clone {repo_https}')
    os.chdir(repo_root)
else:
    repo_root = Path.cwd()

print('cwd ->', os.getcwd())

src_path = str(repo_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print('Installing package with Colab extras...')
get_ipython().system('python -m pip -q install -e .[colab]')
print('Install complete.')


In [ ]:
# Week 1-2 / Step 2: configure DVC + MLflow roots

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

if IN_COLAB:
    dvc_remote_path = Path('/content/drive/MyDrive/rlhf_logic_verification/dvc')
    mlflow_root = Path('/content/drive/MyDrive/rlhf_logic_verification/mlruns')
else:
    dvc_remote_path = repo_root / 'artifacts' / 'local_dvc'
    mlflow_root = repo_root / 'mlruns'

dvc_remote_path.mkdir(parents=True, exist_ok=True)
mlflow_root.mkdir(parents=True, exist_ok=True)
os.environ['MLFLOW_TRACKING_URI'] = mlflow_root.resolve().as_uri()

if IN_COLAB:
    get_ipython().system('dvc init || true')
    get_ipython().system(f'dvc remote add -d colab_drive {dvc_remote_path} || true')

print({'dvc_remote_path': str(dvc_remote_path), 'mlflow_root': str(mlflow_root), 'mlflow_uri': os.environ['MLFLOW_TRACKING_URI']})


In [ ]:
# Week 1-2 / Step 3: run arithmetic demo and inspect outputs

from verifier_guided_reasoning.pipeline import run_small_demo

arith_summary = run_small_demo(
    output_path='artifacts/eval/demo_summary.json',
    report_markdown_path='artifacts/eval/demo_report.md',
    tracker_root=str(mlflow_root),
)

print('Arithmetic metrics:')
pprint({k: arith_summary[k] for k in ['num_traces', 'final_accuracy', 'pass_rate', 'error_counts']})

if pd is not None:
    display(pd.DataFrame(arith_summary.get('traces', [])))

report_path = Path('artifacts/eval/demo_report.md')
print('
Report path:', report_path)
print(report_path.read_text(encoding='utf-8')[:900])


## Weeks 3-4: Data Curation + Reviewer Controls + Export

Goal: run an annotation-style workflow with explicit reviewer actions.


In [ ]:
# Week 3-4 / Step 1: normalize + gate source rows

from verifier_guided_reasoning.datasets import build_demo_rows, normalize_gsm8k_record, gate_trace_record

rows = build_demo_rows()
traces = [normalize_gsm8k_record(row, split='demo') for row in rows]

gated_rows = []
for trace in traces:
    gate = gate_trace_record(trace)
    gated_rows.append({
        'sample_id': trace.sample_id,
        'accepted': gate.accepted,
        'schema_valid': gate.schema_valid,
        'verifier_agreement': gate.verifier_agreement,
        'reasons': ', '.join(gate.reasons) if gate.reasons else '',
        'question_preview': trace.question[:90] + '...'
    })

if pd is not None:
    display(pd.DataFrame(gated_rows))
else:
    pprint(gated_rows)


In [ ]:
# Week 3-4 / Step 2: generate candidates + build review records

from verifier_guided_reasoning.generation import build_mock_generator, generate_candidates
from verifier_guided_reasoning.review import build_review_records

candidates = generate_candidates(
    rows=rows,
    generator=build_mock_generator(),
    model_name='mock-candidate-generator',
    split='train',
)
review_records = build_review_records(candidates, model_name='mock-candidate-generator', split='train')

preview_rows = [
    {
        'prompt_id': r.prompt_id,
        'candidate_id': r.candidate_id,
        'verifier_pass': r.verifier_pass,
        'verifier_score': round(r.verifier_score, 3),
        'raw_output_preview': r.raw_output[:80].replace('
', ' ') + '...'
    }
    for r in review_records
]
if pd is not None:
    display(pd.DataFrame(preview_rows))
else:
    pprint(preview_rows)


In [ ]:
# Week 3-4 / Step 3: reviewer UI (accept/reject/fix + notes + popup)

from IPython.display import display, HTML, Javascript

try:
    import ipywidgets as widgets
except Exception as exc:
    raise RuntimeError('ipywidgets required. Run: pip install ipywidgets') from exc

from verifier_guided_reasoning.review import summarize_review_metrics

by_prompt = defaultdict(list)
for rec in review_records:
    by_prompt[rec.prompt_id].append(rec)
for prompt_id in by_prompt:
    by_prompt[prompt_id] = sorted(by_prompt[prompt_id], key=lambda x: x.verifier_score, reverse=True)

controls = {}
widget_blocks = []

for prompt_id, group in by_prompt.items():
    widget_blocks.append(widgets.HTML(value=f"<h4>Prompt {prompt_id}</h4><div>{group[0].prompt}</div>"))
    for rec in group:
        action = widgets.Dropdown(
            options=['reject', 'accept', 'fix', 'needs_second_review'],
            value='reject',
            description='Label:',
            layout=widgets.Layout(width='260px'),
        )
        notes = widgets.Textarea(
            value='',
            placeholder='Why this action?',
            description='Notes:',
            layout=widgets.Layout(width='620px', height='70px'),
        )
        score = widgets.FloatSlider(
            value=float(rec.verifier_score),
            min=0,
            max=1,
            step=0.05,
            description='Score:',
            layout=widgets.Layout(width='430px'),
        )

        errors = sorted({vr.get('error_type') for vr in rec.verifier_results if vr.get('error_type')})
        err = ', '.join(errors) if errors else 'none'
        widget_blocks.append(
            widgets.HTML(
                value=(
                    "<div style='border:1px solid #ddd;padding:10px;border-radius:8px;margin:6px 0;background:#fafafa;'>"
                    f"<b>{rec.candidate_id}</b> | verifier_score=<b>{rec.verifier_score:.2f}</b> | errors=<code>{err}</code>"
                    f"<details><summary>Raw output</summary><pre style='white-space:pre-wrap;'>{rec.raw_output}</pre></details>"
                    "</div>"
                )
            )
        )
        widget_blocks.append(widgets.HBox([action, score]))
        widget_blocks.append(notes)
        controls[rec.review_id] = {'record': rec, 'action': action, 'notes': notes, 'score': score}

save_btn = widgets.Button(description='Save Labels', button_style='success', icon='check')
preview_btn = widgets.Button(description='Preview Labels', button_style='info', icon='table')
out = widgets.Output()


def _apply_labels():
    for ui in controls.values():
        rec = ui['record']
        rec.curator_action = ui['action'].value
        rec.curator_score = float(ui['score'].value)
        rec.curator_notes = ui['notes'].value.strip() or None


def _on_preview(_):
    _apply_labels()
    rows = [
        {
            'review_id': rec.review_id,
            'prompt_id': rec.prompt_id,
            'candidate_id': rec.candidate_id,
            'verifier_score': rec.verifier_score,
            'curator_action': rec.curator_action,
            'curator_score': rec.curator_score,
            'curator_notes': rec.curator_notes or '',
        }
        for rec in review_records
    ]
    with out:
        out.clear_output(wait=True)
        if pd is not None:
            display(pd.DataFrame(rows).sort_values(['prompt_id', 'verifier_score'], ascending=[True, False]))
        else:
            pprint(rows)


def _on_save(_):
    _apply_labels()
    metrics = summarize_review_metrics(review_records)
    with out:
        out.clear_output(wait=True)
        display(HTML(
            "<div style='padding:10px;background:#e8f5e9;border:1px solid #66bb6a;border-radius:8px;'>"
            "<b>Labels saved.</b><br>"
            f"Acceptance rate: {metrics['acceptance_rate']:.2%} | "
            f"Disagreement: {metrics['curator_verifier_disagreement_rate']:.2%}"
            "</div>"
        ))
        pprint(metrics)
    try:
        display(Javascript("alert('Week 3-4 labels saved. Ready for export.')"))
    except Exception:
        pass

preview_btn.on_click(_on_preview)
save_btn.on_click(_on_save)

display(widgets.HBox([preview_btn, save_btn]))
display(widgets.VBox(widget_blocks))
display(out)


In [ ]:
# Week 3-4 / Step 4: export curated artifacts

from verifier_guided_reasoning.datasets import write_jsonl
from verifier_guided_reasoning.review import write_review_jsonl, export_sft_examples, export_preference_pairs, summarize_review_metrics

curation_dir = Path('artifacts/curation')
curation_dir.mkdir(parents=True, exist_ok=True)

review_path = curation_dir / 'review_batch_annotated.jsonl'
sft_path = curation_dir / 'sft_curated.jsonl'
prefs_path = curation_dir / 'preferences_curated.jsonl'

write_review_jsonl(review_records, review_path)
sft_examples = export_sft_examples(review_records)
pref_pairs = export_preference_pairs(review_records, min_margin=0.0)
write_jsonl(sft_examples, sft_path)
write_jsonl([p.to_dict() for p in pref_pairs], prefs_path)

review_metrics = summarize_review_metrics(review_records)
pprint({
    'review_records': len(review_records),
    'sft_examples': len(sft_examples),
    'preference_pairs': len(pref_pairs),
    'acceptance_rate': review_metrics['acceptance_rate'],
    'artifacts': [str(review_path), str(sft_path), str(prefs_path)],
})


In [ ]:
# Week 3-4 / Step 5: SFT configuration snapshot

TRAINING_CONFIG = {
    'base_model': 'Qwen/Qwen2.5-1.5B-Instruct',
    'comparison_model': 'Qwen/Qwen2.5-Math-1.5B-Instruct',
    'tuning_method': 'qlora',
    'max_seq_length': 2048,
    'datasets': {
        'sft_input': 'artifacts/curation/sft_curated.jsonl',
        'preference_input': 'artifacts/curation/preferences_curated.jsonl',
        'benchmark': 'openai/gsm8k',
    },
}

pprint(TRAINING_CONFIG)


## Weeks 5-6: Candidate Selection + Diagnostics (Implemented)


In [ ]:
# Week 5-6 / Step 1: best-of-N reranking analysis

prompt_groups = defaultdict(list)
for rec in review_records:
    prompt_groups[rec.prompt_id].append(rec)

comparison_rows = []
for prompt_id, group in prompt_groups.items():
    group_sorted = sorted(group, key=lambda x: x.candidate_id)
    baseline = group_sorted[0]
    best = max(group, key=lambda x: x.verifier_score)
    comparison_rows.append({
        'prompt_id': prompt_id,
        'baseline_candidate': baseline.candidate_id,
        'baseline_score': baseline.verifier_score,
        'baseline_pass': baseline.verifier_pass,
        'best_candidate': best.candidate_id,
        'best_score': best.verifier_score,
        'best_pass': best.verifier_pass,
        'score_gain': best.verifier_score - baseline.verifier_score,
    })

if pd is not None:
    display(pd.DataFrame(comparison_rows))
else:
    pprint(comparison_rows)

baseline_pass_rate = sum(1 for row in comparison_rows if row['baseline_pass']) / max(len(comparison_rows), 1)
best_pass_rate = sum(1 for row in comparison_rows if row['best_pass']) / max(len(comparison_rows), 1)
print({'baseline_pass_rate': baseline_pass_rate, 'best_of_n_pass_rate': best_pass_rate})


In [ ]:
# Week 5-6 / Step 2: rejection sampling threshold UI

import ipywidgets as widgets

threshold = widgets.FloatSlider(value=0.5, min=0, max=1, step=0.05, description='Min score:')
apply_btn = widgets.Button(description='Apply', button_style='primary')
out = widgets.Output()


def _apply(_):
    th = float(threshold.value)
    kept = [rec for rec in review_records if rec.verifier_score >= th]
    rejected = [rec for rec in review_records if rec.verifier_score < th]
    with out:
        out.clear_output(wait=True)
        pprint({'threshold': th, 'kept': len(kept), 'rejected': len(rejected), 'keep_rate': len(kept)/max(len(review_records),1)})

        rows = [{'candidate_id': r.candidate_id, 'prompt_id': r.prompt_id, 'score': round(r.verifier_score,3)} for r in kept]
        if pd is not None:
            display(pd.DataFrame(rows))
        else:
            pprint(rows)

apply_btn.on_click(_apply)

display(widgets.HBox([threshold, apply_btn]))
display(out)


In [ ]:
# Week 5-6 / Step 3: failure-class diagnostics

error_counter = Counter()
for rec in review_records:
    for vr in rec.verifier_results:
        if vr.get('error_type'):
            error_counter[vr['error_type']] += 1

diag_rows = [{'error_type': et, 'count': c} for et, c in error_counter.most_common()]
if pd is not None:
    display(pd.DataFrame(diag_rows))
else:
    pprint(diag_rows)

if plt is not None and diag_rows:
    plt.figure(figsize=(8,4))
    plt.bar([r['error_type'] for r in diag_rows], [r['count'] for r in diag_rows])
    plt.xticks(rotation=25, ha='right')
    plt.title('Week 5-6 Failure Types')
    plt.tight_layout()
    plt.show()


## Weeks 7-8: Logic Extension + Publication Gate (Implemented)

Goal:
- add a narrow logic verifier extension,
- keep logic metrics separate from arithmetic metrics,
- enforce publication readiness with artifact hashes and a dataset card.


In [ ]:
# Week 7-8 / Step 1: run separate logic benchmark demo

from verifier_guided_reasoning.logic_pipeline import run_logic_demo

logic_summary = run_logic_demo(
    output_path='artifacts/eval/logic_summary.json',
    tracker_root=str(mlflow_root),
)

print('Logic metrics:')
pprint({k: logic_summary[k] for k in ['num_traces', 'logic_label_accuracy', 'pass_rate', 'error_counts']})
if pd is not None:
    display(pd.DataFrame(logic_summary.get('traces', [])))


In [ ]:
# Week 7-8 / Step 2: keep arithmetic and logic reporting explicitly separate

arith_panel = {
    'task': 'arithmetic',
    'num_traces': arith_summary.get('num_traces'),
    'final_accuracy': arith_summary.get('final_accuracy'),
    'pass_rate': arith_summary.get('pass_rate'),
    'top_errors': arith_summary.get('error_counts', {}),
}

logic_panel = {
    'task': 'logic_entailment',
    'num_traces': logic_summary.get('num_traces'),
    'logic_label_accuracy': logic_summary.get('logic_label_accuracy'),
    'pass_rate': logic_summary.get('pass_rate'),
    'top_errors': logic_summary.get('error_counts', {}),
}

print('Arithmetic panel (separate):')
pprint(arith_panel)
print('
Logic panel (separate):')
pprint(logic_panel)

if pd is not None:
    display(pd.DataFrame([
        {'task': 'arithmetic', 'accuracy_metric': arith_panel['final_accuracy'], 'pass_rate': arith_panel['pass_rate']},
        {'task': 'logic_entailment', 'accuracy_metric': logic_panel['logic_label_accuracy'], 'pass_rate': logic_panel['pass_rate']},
    ]))


In [ ]:
# Week 7-8 / Step 3: reproducibility gate + dataset card generation

from verifier_guided_reasoning.publishing import build_artifact_manifest, reproducibility_gate, write_dataset_card
from IPython.display import Javascript

publish_dir = Path('artifacts/publish')
publish_dir.mkdir(parents=True, exist_ok=True)

artifact_paths = [
    Path('artifacts/eval/demo_summary.json'),
    Path('artifacts/eval/logic_summary.json'),
    Path('artifacts/curation/sft_curated.jsonl'),
    Path('artifacts/curation/preferences_curated.jsonl'),
]
manifest = build_artifact_manifest(artifact_paths)
repro_ok, repro_issues = reproducibility_gate(manifest)

if pd is not None:
    display(pd.DataFrame(manifest))
else:
    pprint(manifest)

combined_metrics = {
    'arithmetic_final_accuracy': float(arith_summary.get('final_accuracy', 0.0)),
    'logic_label_accuracy': float(logic_summary.get('logic_label_accuracy', 0.0)),
    'curation_acceptance_rate': float(review_metrics.get('acceptance_rate', 0.0)),
}

card_path = write_dataset_card(
    output_path=publish_dir / 'dataset_card.md',
    title='Verifier-Guided Curated Arithmetic + Logic Dataset',
    overview='Curated reasoning traces with deterministic arithmetic and logic verification for data-quality-aware model improvement.',
    metrics=combined_metrics,
    manifest=manifest,
    release_notes=[
        'Arithmetic and logic metrics are intentionally tracked in separate sections.',
        'Publish only when reproducibility gate passes.',
    ] + repro_issues,
)

print({'reproducibility_gate_passed': repro_ok, 'dataset_card': str(card_path)})
print(card_path.read_text(encoding='utf-8')[:1200])

try:
    display(Javascript("alert('Week 7-8 publication gate complete. Check reproducibility status before publishing.')"))
except Exception:
    pass


In [ ]:
# Week 7-8 / Step 4: publication checklist panel for employer-facing demo

publish_checks = [
    {'check': 'Arithmetic evaluation artifact exists', 'passed': Path('artifacts/eval/demo_summary.json').exists()},
    {'check': 'Logic evaluation artifact exists', 'passed': Path('artifacts/eval/logic_summary.json').exists()},
    {'check': 'Curated SFT export exists', 'passed': Path('artifacts/curation/sft_curated.jsonl').exists()},
    {'check': 'Curated preference export exists', 'passed': Path('artifacts/curation/preferences_curated.jsonl').exists()},
    {'check': 'Reproducibility gate passed', 'passed': repro_ok},
]

all_passed = all(item['passed'] for item in publish_checks)

if pd is not None:
    display(pd.DataFrame(publish_checks))
else:
    pprint(publish_checks)

print('Publication readiness:', 'READY' if all_passed else 'NOT READY')


In [ ]:
print('Notebook status: Weeks 1-8 implemented with arithmetic + logic separation and publication gate checks.')
